# Demo: SEC EDGAR Data Pipeline

**Module:** `src.common.ingestion`  
**Purpose:** Download 10-K filings from SEC EDGAR, extract key sections, and save as structured Markdown.  
**Shared by:** All 4 RAG systems (ensures fair comparison).

### Why Markdown?

Markdown was chosen as the **unified data format** for all 4 systems to minimize format-induced bias:
- **RAG systems** (1 & 2): `##` headers enable semantic chunking (split at section boundaries)
- **Long-Context systems** (3 & 4): Minimal token overhead vs. JSON syntax
- **Neutral**: Doesn't advantage either architecture class

---

## 1. Setup & Configuration

The pipeline uses `configs/base.yaml` for shared settings and `.env` for secrets (API keys, SEC identity).  
All config is loaded through `src.common.config.load_config()`.

In [ ]:
import logging
from src.common import load_config, setup_logging

setup_logging(logging.INFO)

# Show current config (secrets are injected from .env)
config = load_config()
print("LLM Model:", config['llm']['model'])
print("Embedding Model:", config['embedding']['model'])
print("Target Companies:", [c['ticker'] for c in config['companies']])

## 2. Download a Single 10-K Filing

The `download_filing()` function:
1. Connects to SEC EDGAR via `edgartools`
2. Finds the latest 10-K filing for a given ticker
3. Extracts the full text and individual sections (Items 1-15)
4. Saves:
   - `data/raw/{TICKER}/10K_{date}.txt` — Raw text from EDGAR
   - `data/processed/{TICKER}/10K_{date}.md` — Structured Markdown with `##` section headers
   - `data/processed/{TICKER}/10K_{date}.meta.json` — Machine-readable metadata sidecar

In [ ]:
from src.common import download_filing

# Download Apple's latest 10-K
filing = download_filing("AAPL")

print(f"Company: {filing.metadata.company_name}")
print(f"Filing Date: {filing.metadata.filing_date}")
print(f"CIK: {filing.metadata.cik}")
print(f"Full Text Length: {len(filing.full_text):,} characters")
print(f"Sections Extracted: {len(filing.sections)}")

## 3. Inspect the Markdown Output

The generated Markdown has this structure:
```
# Apple Inc. — 10-K Annual Report
**Ticker:** AAPL
...
---
## Business
(content)
## Risk Factors
(content)
## MD&A
(content)
```

This structure enables **semantic chunking** by headers for RAG systems.

In [ ]:
# Show all extracted sections and their sizes
print("Extracted Sections:")
print("-" * 50)
for section_name, content in filing.sections.items():
    print(f"  {section_name:45s} | {len(content):>8,} chars")
print("-" * 50)
print(f"  {'TOTAL':45s} | {sum(len(c) for c in filing.sections.values()):>8,} chars")

In [ ]:
# Preview the generated Markdown (first 800 chars)
markdown_output = filing.to_markdown()
print("=== MARKDOWN OUTPUT (Preview) ===")
print(markdown_output[:800])
print("...")
print(f"\nTotal Markdown length: {len(markdown_output):,} chars")

## 4. Load a Previously Processed Filing

Once downloaded, filings are persisted as `.md` + `.meta.json` in `data/processed/`.  
They can be loaded without re-downloading from EDGAR.

In [ ]:
from src.common import load_processed_filing

# Load from disk (no network call)
cached = load_processed_filing("AAPL")
print(f"Loaded: {cached.metadata.company_name} ({cached.metadata.filing_date})")
print(f"Sections: {list(cached.sections.keys())}")

## 5. Batch Download All Configured Companies

The `download_all_filings()` function processes all companies defined in `configs/base.yaml`.  
Currently configured: **AAPL, MSFT, AMZN**.

In [ ]:
from src.common import download_all_filings

# Download all 3 companies
results = download_all_filings()

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for f in results:
    print(f"  {f.metadata.ticker:6s} | {f.metadata.company_name:30s} | {len(f.sections)} sections | {len(f.full_text):>8,} chars")